# Credit Risk Portfolio Stress Testing

## Management Summary

The portfolio contains 10,000 simulated loans with total exposure of roughly Rs 488 Cr. Baseline expected loss is about Rs 9.4 Cr, while severe stress expected loss rises to about Rs 40.3 Cr. The severe scenario creates an incremental capital shortfall of roughly Rs 30.8 Cr. Losses are concentrated in lower-rated BB/B borrowers and high-LTV watchlist loans. Management should prioritize watchlist monitoring, collateral review, and exposure limits in stressed sectors.

## 1. Setup

This notebook imports the reusable model functions from `src/credit_risk_portfolio.py`. Run from the project root so the relative paths resolve cleanly.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from credit_risk_portfolio import (
    BuildConfig,
    build_scenario_results,
    build_segment_tables,
    generate_portfolio,
    inr_cr,
    load_lendingclub_recovery_benchmark,
    run_build,
)

import pandas as pd

## 2. Generate Portfolio

The portfolio is synthetic but calibrated to a bank/NBFC-style secured and unsecured loan book. A fixed seed makes the results reproducible.

In [ ]:
portfolio = generate_portfolio(BuildConfig(seed=42, loan_count=10_000))
portfolio.head()

In [ ]:
portfolio_summary = pd.DataFrame({
    'Metric': ['Loans', 'Portfolio EAD Rs Cr', 'Baseline EL Rs Cr', 'Watchlist Loans'],
    'Value': [
        len(portfolio),
        inr_cr(portfolio['EAD'].sum()),
        inr_cr(portfolio['Expected_Loss'].sum()),
        int(portfolio['downgrade_risk'].sum()),
    ]
})
portfolio_summary

## 3. PD, LGD, EAD and Expected Loss

Expected loss is calculated as `PD x LGD x EAD`. The rating-level PD table is based on common rating-agency style historical default assumptions, while LGD is set by rating severity.

In [ ]:
rating_view = (
    portfolio.groupby('credit_rating', observed=False)
    .agg(loans=('loan_id', 'count'), ead=('EAD', 'sum'), avg_pd=('PD', 'mean'), avg_lgd=('LGD', 'mean'), expected_loss=('Expected_Loss', 'sum'))
    .assign(ead_cr=lambda x: x['ead'].map(inr_cr), expected_loss_cr=lambda x: x['expected_loss'].map(inr_cr))
)
rating_view

## 4. Stress Testing

Three macro scenarios are used: Baseline, Downside and Severe. The stress changes PD by rating and keeps LGD constant for interpretability.

In [ ]:
scenario_df, scenario_loan_level = build_scenario_results(portfolio)
scenario_df.assign(Total_Portfolio_Cr=scenario_df['Total_Portfolio'].map(inr_cr), Expected_Loss_Cr=scenario_df['Expected_Loss'].map(inr_cr), Capital_Required_Cr=scenario_df['Capital_Required'].map(inr_cr))

In [ ]:
baseline_el = scenario_df.loc[scenario_df['Scenario'] == 'Baseline', 'Expected_Loss'].iloc[0]
severe_el = scenario_df.loc[scenario_df['Scenario'] == 'Severe', 'Expected_Loss'].iloc[0]
capital_shortfall = severe_el - baseline_el
print(f"Capital shortfall under severe stress: Rs {inr_cr(capital_shortfall):,.1f} Cr")
print(f"Expected loss uplift: {severe_el / baseline_el:,.1f}x")

## 5. Segment Risk and Watchlist

The watchlist flags lower-rated loans with high LTV and repayment pressure. These are the loans a portfolio manager would monitor first under stress.

In [ ]:
segments = build_segment_tables(portfolio, scenario_loan_level)
segments['sector_summary'].assign(ead_cr=lambda x: x['ead'].map(inr_cr), baseline_el_cr=lambda x: x['baseline_el'].map(inr_cr), severe_el_cr=lambda x: x['severe_el'].map(inr_cr))

In [ ]:
segments['watchlist_top100'][['loan_id', 'sector', 'credit_rating', 'loan_amount', 'ltv_ratio', 'dti_ratio', 'PD', 'LGD', 'Severe_EL']].head(10)

## 6. LendingClub Recovery Benchmark

The supplied LendingClub defaults dataset is used as a real-data benchmark for recovery and credit conversion behavior. It is default-only data, so it supports LGD context rather than full portfolio PD training.

In [ ]:
load_lendingclub_recovery_benchmark()

## 7. Export Power BI Dataset

The build exports CSV files, a dashboard-ready Excel workbook, and a dashboard preview image into `outputs/`.

In [ ]:
run_build()